In [1]:
# Core data libraries
import pandas as pd
import numpy as np

# For plotting quick checks (optional)
import matplotlib.pyplot as plt
import seaborn as sns

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


In [4]:

forecast_df = pd.read_csv("MIG_8week_forecast.csv")
df_features = pd.read_csv("../data/MIG_features.csv")   # or your actual file name

print("forecast_df shape:", forecast_df.shape)
print("df_features shape:", df_features.shape)

forecast_df.head(), df_features.head()

forecast_df shape: (1680, 6)
df_features shape: (30210, 32)


(    site_id        date    sarimax    xgboost  random_forest   ensemble
 0  SITE_001  2025-01-01  58.846242  52.647118       48.30653  53.266630
 1  SITE_001  2025-01-02  59.494834  52.647118       48.30653  53.482827
 2  SITE_001  2025-01-03  59.559194  52.647118       48.30653  53.504280
 3  SITE_001  2025-01-04  59.565581  52.647118       48.30653  53.506409
 4  SITE_001  2025-01-05  59.566214  52.647118       48.30653  53.506621,
          date   site_id cement_type  planned_pour_tonnes  consumed_tonnes  opening_inventory_tonnes  deliveries_tonnes  closing_inventory_tonnes  rain_mm  avg_temp_c  expected_closing region  \
 0  2022-03-31  SITE_001     CEM_III                46.10            19.44                      0.00              19.44                      0.00     4.55       20.20              0.00  North   
 1  2022-04-01  SITE_001      CEM_II                31.10            31.10                      0.00              38.53                      7.43     7.13       28.90     

In [5]:
# 3. Clamp negative forecasts to zero (physical constraint)

cols_to_clip = ['ensemble', 'sarimax', 'xgboost', 'random_forest']

forecast_df[cols_to_clip] = (
    forecast_df[cols_to_clip]
    .clip(lower=0)
)

forecast_df[cols_to_clip].describe()


,ensemble,sarimax,xgboost,random_forest
count,1680.000000,1680.000000,1680.000000,1680.000000
mean,21.517528,21.403756,21.477191,22.595205
std,15.449205,18.062471,16.260871,12.648510
min,0.000000,0.000000,0.472665,4.973082
25%,7.069403,5.441660,6.957800,11.637020
50%,19.131071,17.684857,17.499052,17.772252
75%,32.161310,33.239848,35.407940,34.026454
max,53.506644,65.425851,53.175920,49.097641


In [6]:
# 4. Build inventory snapshot: last known state per site

# Ensure features are sorted by date
df_features = df_features.sort_values(['site_id', 'date'])

# Take last row per site as the starting point
inventory_snapshot = (
    df_features
    .groupby('site_id')
    .tail(1)[[
        'site_id',
        'date',
        'closing_inventory_tonnes',
        'silo_capacity'
    ]]
)

inventory_snapshot.rename(columns={
    'date': 'last_hist_date',
    'closing_inventory_tonnes': 'last_closing_inventory'
}, inplace=True)

inventory_snapshot.head()


,site_id,last_hist_date,last_closing_inventory,silo_capacity
1006,SITE_001,2024-12-31,5.85,448
2013,SITE_002,2024-12-31,19963.37,288
3020,SITE_003,2024-12-31,31.00,314
4027,SITE_004,2024-12-31,20505.34,472
5034,SITE_005,2024-12-31,32.85,230


In [7]:
# 5. Merge forecast with inventory snapshot

forecast_inv = forecast_df.merge(
    inventory_snapshot,
    on='site_id',
    how='left'
)

forecast_inv.head()


,site_id,date,sarimax,xgboost,random_forest,ensemble,last_hist_date,last_closing_inventory,silo_capacity
0,SITE_001,2025-01-01,58.846242,52.647118,48.30653,53.266630,2024-12-31,5.85,448
1,SITE_001,2025-01-02,59.494834,52.647118,48.30653,53.482827,2024-12-31,5.85,448
2,SITE_001,2025-01-03,59.559194,52.647118,48.30653,53.504280,2024-12-31,5.85,448
3,SITE_001,2025-01-04,59.565581,52.647118,48.30653,53.506409,2024-12-31,5.85,448
4,SITE_001,2025-01-05,59.566214,52.647118,48.30653,53.506621,2024-12-31,5.85,448


In [8]:
# 6. Add simple future deliveries assumption

# For now: assume no future deliveries (you can refine later)
forecast_inv['deliveries_tonnes'] = 0.0


In [9]:
# 7. Simulate inventory day-by-day per site

inventory_results = []

for site, df_site in forecast_inv.groupby('site_id'):
    df_site = df_site.sort_values('date').copy()

    # Starting inventory = last closing inventory from history
    current_inventory = df_site['last_closing_inventory'].iloc[0]

    opening_list = []
    closing_list = []

    for _, row in df_site.iterrows():
        # Opening inventory for this day
        opening_list.append(current_inventory)

        # Inventory equation:
        # closing = opening + deliveries - forecasted consumption (ensemble)
        closing = (
            current_inventory +
            row['deliveries_tonnes'] -
            row['ensemble']
        )

        # Clamp between 0 and silo capacity
        closing = max(0, min(closing, row['silo_capacity']))

        closing_list.append(closing)

        # Next day's opening = today's closing
        current_inventory = closing

    df_site['opening_inventory_sim'] = opening_list
    df_site['closing_inventory_sim'] = closing_list

    inventory_results.append(df_site)

inventory_future = pd.concat(inventory_results)
inventory_future = inventory_future.sort_values(['site_id', 'date'])

inventory_future.head()


,site_id,date,sarimax,xgboost,random_forest,ensemble,last_hist_date,last_closing_inventory,silo_capacity,deliveries_tonnes,opening_inventory_sim,closing_inventory_sim
0,SITE_001,2025-01-01,58.846242,52.647118,48.30653,53.266630,2024-12-31,5.85,448,0.0,5.85,0.0
1,SITE_001,2025-01-02,59.494834,52.647118,48.30653,53.482827,2024-12-31,5.85,448,0.0,0.00,0.0
2,SITE_001,2025-01-03,59.559194,52.647118,48.30653,53.504280,2024-12-31,5.85,448,0.0,0.00,0.0
3,SITE_001,2025-01-04,59.565581,52.647118,48.30653,53.506409,2024-12-31,5.85,448,0.0,0.00,0.0
4,SITE_001,2025-01-05,59.566214,52.647118,48.30653,53.506621,2024-12-31,5.85,448,0.0,0.00,0.0


In [10]:
# 8. Add reorder alerts and stockout risk

# Reorder threshold: e.g. 25% of silo capacity
reorder_point_ratio = 0.25

inventory_future['reorder_alert'] = (
    inventory_future['closing_inventory_sim'] <
    inventory_future['silo_capacity'] * reorder_point_ratio
)

inventory_future['stockout_risk'] = (
    inventory_future['closing_inventory_sim'] <= 0
)

inventory_future[['site_id', 'date', 'closing_inventory_sim',
                  'reorder_alert', 'stockout_risk']].head()


,site_id,date,closing_inventory_sim,reorder_alert,stockout_risk
0,SITE_001,2025-01-01,0.0,True,True
1,SITE_001,2025-01-02,0.0,True,True
2,SITE_001,2025-01-03,0.0,True,True
3,SITE_001,2025-01-04,0.0,True,True
4,SITE_001,2025-01-05,0.0,True,True


In [11]:
# 9. Build dashboard-ready dataset

dashboard_df = inventory_future[[
    'site_id', 'date',
    'ensemble', 'sarimax', 'xgboost', 'random_forest',
    'opening_inventory_sim', 'closing_inventory_sim',
    'silo_capacity', 'deliveries_tonnes',
    'reorder_alert', 'stockout_risk'
]].copy()

dashboard_df.head()


,site_id,date,ensemble,sarimax,xgboost,random_forest,opening_inventory_sim,closing_inventory_sim,silo_capacity,deliveries_tonnes,reorder_alert,stockout_risk
0,SITE_001,2025-01-01,53.266630,58.846242,52.647118,48.30653,5.85,0.0,448,0.0,True,True
1,SITE_001,2025-01-02,53.482827,59.494834,52.647118,48.30653,0.00,0.0,448,0.0,True,True
2,SITE_001,2025-01-03,53.504280,59.559194,52.647118,48.30653,0.00,0.0,448,0.0,True,True
3,SITE_001,2025-01-04,53.506409,59.565581,52.647118,48.30653,0.00,0.0,448,0.0,True,True
4,SITE_001,2025-01-05,53.506621,59.566214,52.647118,48.30653,0.00,0.0,448,0.0,True,True


In [12]:
# 10. Save dashboard dataset for Notebook 07 (Dashboard)

dashboard_df.to_csv("MIG_dashboard_inventory_forecast.csv", index=False)
print("Saved MIG_dashboard_inventory_forecast.csv successfully.")


Saved MIG_dashboard_inventory_forecast.csv successfully.
